# RiverMind — train the waste detector (YOLO11n, v1)

Runs on **Google Colab with a free T4 GPU**. It:
1. downloads the Mendeley dataset *Floating Waste and Aquatic Vegetation Image Dataset from Turbid Rivers* v2
   (DOI 10.17632/j26w4m645z.2, CC BY 4.0) and checks its SHA-256,
2. renames its classes to the CONTRACT.md names (`floating_waste` → `other`, `river_vegetation` → `vegetation`),
3. trains YOLO11n, evaluates on the dataset's held-out **test** split, and exports ONNX (≤ 25 MB),
4. runs a zero-shot **YOLO-World baseline** on the same test split for comparison.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all** and allow Google Drive
access. Keep the tab open. Everything is saved to `MyDrive/rivermind/`, so if Colab disconnects just **Run all**
again — training resumes from the last epoch.

**When it finishes,** download `MyDrive/rivermind/export/` (the `.onnx`, `.pt`, `metrics_test.json`,
`baseline_test.json`, `results.csv`).

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(gpu or "NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again")

In [ ]:
%pip install -q ultralytics==8.4.43

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/rivermind")
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)

DATASET_URL = ("https://data.mendeley.com/public-files/datasets/j26w4m645z/files/"
               "fe81eec3-65c8-48c3-809b-9a9a3a359c82/file_downloaded")
DATASET_SHA256 = "e43166dde54f82fa9dc00adc5f3943ea7a9083f4beda1490a774fcea6dc7d8b3"
RUN_NAME = "yolo11n-rivermind-v1"
IMGSZ = 960      # objects are small (median box ~0.6% of the image), so train above the 640 default
EPOCHS = 60
PATIENCE = 15
MAX_SIDE = 1920  # some photos are 12000 px wide; shrink them once so the data loader keeps up
SEED = 0

In [ ]:
import hashlib
import subprocess

# Mendeley answers 403 to Python's default user agent, so download with curl and a browser user agent.
# If Colab is still refused, upload the zip to MyDrive/rivermind/ and set: zip_path = DRIVE / "turbid-rivers-v2.zip"
zip_path = WORK / "turbid-rivers-v2.zip"
if not zip_path.exists():
    print("downloading 1.76 GB from Mendeley ...")
    subprocess.run(["curl", "-L", "--fail", "-sS", "-A", "Mozilla/5.0", "-o", str(zip_path), DATASET_URL], check=True)
digest = hashlib.sha256()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        digest.update(chunk)
assert digest.hexdigest() == DATASET_SHA256, "checksum mismatch: delete /content/work and Run all again"
print("dataset checksum OK")

In [ ]:
import shutil
import zipfile

from PIL import Image

SRC = WORK / "src"
DATA = WORK / "data"
if not SRC.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(SRC)
root = SRC / "Floating Waste and River Vegetation Dataset"

# Label ids stay 0 and 1; only the names change to the CONTRACT.md classes. The dataset has no bottle/bag split.
NAMES = ["other", "vegetation"]
EXPECTED = {"train": 1104, "val": 315, "test": 158}

shutil.rmtree(DATA, ignore_errors=True)
skipped, shrunk = [], 0
for split, expected in EXPECTED.items():
    images = sorted((root / split / "images").glob("*.jpg"))
    assert len(images) == expected, f"{split}: expected {expected} images, found {len(images)}"
    (DATA / split / "images").mkdir(parents=True)
    (DATA / split / "labels").mkdir(parents=True)
    for img_path in images:
        label = root / split / "labels" / f"{img_path.stem}.txt"
        dst = DATA / split / "images" / img_path.name
        im = Image.open(img_path)
        if im.getexif().get(0x0112, 1) != 1:  # a rotation tag: its boxes may not line up with the pixels
            skipped.append(f"{split}/{img_path.name}")
            continue
        if max(im.size) > MAX_SIDE:
            im.thumbnail((MAX_SIDE, MAX_SIDE))  # boxes are normalised 0-1, so the labels stay valid
            im.convert("RGB").save(dst, quality=95)
            shrunk += 1
        else:
            shutil.copy(img_path, dst)
        shutil.copy(label, DATA / split / "labels" / label.name)
print(f"shrunk {shrunk} large photos; skipped {len(skipped)} rotated photos: {skipped}")

yaml_path = DATA / "data.yaml"
yaml_path.write_text(f"path: {DATA}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n"
                     + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
print(yaml_path.read_text())

In [ ]:
from ultralytics import YOLO

runs = DRIVE / "runs"
last = runs / RUN_NAME / "weights" / "last.pt"
if last.exists():  # Colab disconnected mid-run: resume from the last saved epoch
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as finished:  # the run already completed; nothing to resume
        print(finished)
else:
    YOLO("yolo11n.pt").train(
        data=str(yaml_path), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=16, workers=2,
        cache="ram", seed=SEED, deterministic=True, project=str(runs), name=RUN_NAME, exist_ok=True,
    )

In [ ]:
import json

EXPORT = DRIVE / "export"
EXPORT.mkdir(exist_ok=True)
n_test = len(list((DATA / "test" / "images").glob("*.jpg")))


def summary(m, model_name):
    return {
        "model": model_name, "split": "test", "images": n_test, "imgsz": IMGSZ,
        "mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
        "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
        "per_class_mAP50": {NAMES[int(c)]: round(float(v), 4)
                            for c, v in zip(m.box.ap_class_index, m.box.ap50, strict=True)},
    }


best = YOLO(str(runs / RUN_NAME / "weights" / "best.pt"))
test_metrics = summary(best.val(data=str(yaml_path), split="test", imgsz=IMGSZ, batch=16,
                                project=str(runs), name=f"{RUN_NAME}-test", exist_ok=True), RUN_NAME)
(EXPORT / "metrics_test.json").write_text(json.dumps(test_metrics, indent=2))
print(json.dumps(test_metrics, indent=2))

In [ ]:
onnx_path = Path(best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False))
size_mb = onnx_path.stat().st_size / 1e6
assert size_mb <= 25, f"ONNX is {size_mb:.1f} MB, over the 25 MB contract cap"
shutil.copy(onnx_path, EXPORT / f"{RUN_NAME}.onnx")
shutil.copy(runs / RUN_NAME / "weights" / "best.pt", EXPORT / f"{RUN_NAME}.pt")
shutil.copy(runs / RUN_NAME / "results.csv", EXPORT / "results.csv")
print(f"exported {RUN_NAME}.onnx ({size_mb:.1f} MB) to {EXPORT}")

## Baseline: zero-shot YOLO-World on the same test split
No training: the open-vocabulary model is only told the two class names in words. This is the comparison
the PRD asks for ("against the zero-shot baseline").

In [ ]:
world = YOLO("yolov8s-worldv2.pt")
world.set_classes(["floating trash", "aquatic plants"])  # same order as NAMES: other, vegetation
baseline = summary(world.val(data=str(yaml_path), split="test", imgsz=IMGSZ, batch=16,
                             project=str(runs), name="yolo-world-baseline-test", exist_ok=True),
                   "yolov8s-worldv2 zero-shot")
(EXPORT / "baseline_test.json").write_text(json.dumps(baseline, indent=2))
print(json.dumps(baseline, indent=2))
print(f"fine-tuned mAP50 {test_metrics['mAP50']} vs zero-shot {baseline['mAP50']}")